### Importing all required modules:

- BeautifulSoup: Parse html and find elements by class name
- Selenium webdriver: Get source of a page url and simulate clicks for "Show More" buttons
- Time: Used to wait after clicks for page to fully laod entries
- Pandas: To use pandas dataframes, since they can be easily inserted into sql tables
- Sqlite3: To use the sqlite database


In [2]:
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import pandas as pd

import sqlite3
from sqlite3 import Error

### Defining all database functions

In [3]:
def create_connection(path):
    connection = None
    try:
        connection = sqlite3.connect(path)
        print("Connection to sqlite db successful")
    except Error as e:
        print(f"Error '{e}' occurred")
    return connection

In [4]:
def create_database(connection, query):
    cursor = connection.cursor()
    try:
        cursor.execute(query)
        print("Database created successfully")
    except Error as e:
        print(f"Error '{e}' occurred")

In [5]:
def execute_query(connection, query):
    cursor = connection.cursor()
    try:
        cursor.execute(query)
        connection.commit()
        print("Query executed successfully")
    except Error as e:
        print(f"Error '{e}' occurred")

In [6]:
def execute_read_query(connection, query):
    cursor = connection.cursor()
    result = None
    try:
        cursor.execute(query)
        result = cursor.fetchall()
        return result
    except Error as e:
        print(f"Error '{e}' occurred")

In [7]:
def execute_read_query_parameterized(connection, query, parameter):
    cursor = connection.cursor()
    result = None
    try:
        cursor.execute(query, parameter)
        result = cursor.fetchall()
        return result
    except Error as e:
        print(f"Error '{e}' occurred")

### Connecting to the database

In [ ]:
connection = create_connection("database.sqlite")

### Defining all sql queries for creating the databases tables

In [75]:
create_actors_table = """
CREATE TABLE IF NOT EXISTS actors (
    id INTEGER PRIMARY KEY,
    ranking INTEGER,
    name TEXT NOT NULL,
    about TEXT
);
"""
create_movies_table = """
CREATE TABLE IF NOT EXISTS movies (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    rating REAL,
    year INTEGER
);
"""
create_genres_table = """
CREATE TABLE IF NOT EXISTS genres (
    movieId INTEGER,
    genre TEXT,
    PRIMARY KEY (movieId, genre),
    FOREIGN KEY (movieId) REFERENCES movies(id)
);
"""

create_actors_play_movies_table = """
CREATE TABLE IF NOT EXISTS actors_play_movies (
    actorId INTEGER,
    movieId INTEGER,
    PRIMARY KEY (actorId, movieId),
    FOREIGN KEY (actorId) REFERENCES actors(id),
    FOREIGN KEY (movieId) REFERENCES movies(id)
);
"""

create_awards_table = """
CREATE TABLE IF NOT EXISTS awards (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    category TEXT,
    year INTEGER,
    win TEXT,
    actorId INTEGER,
    movieId INTEGER,
    FOREIGN KEY (actorId) REFERENCES actors(id),
    FOREIGN KEY (movieId) REFERENCES movies(id)
);
"""

### Creating the database tables

In [ ]:
execute_query(connection, create_actors_table)
execute_query(connection, create_movies_table)
execute_query(connection, create_genres_table)
execute_query(connection, create_actors_play_movies_table)
execute_query(connection, create_awards_table)

### Scraping the actors data and saving it in a pandas dataframe

Seleniums webdriver module is utilized to get the source of the IMDB page, while
BeautifulSoup is used to parse the html and find all actor list items

I iterate over the list of items. In each iteration the following steps are performed:
1. Find the actors name
2. Find the actor URL and get the actors page
3. Find the about URL and get the about page
4. Find the biography, perform concatenation and formatting
5. Append id, ranking, name and about to lists
6. Increment id and ranking

In [ ]:
from selenium.webdriver.chrome.options import Options
chrome_options = Options()
chrome_options.add_argument('--lang=en')

url = "https://www.imdb.com/list/ls053501318/"
driver = webdriver.Chrome(options=chrome_options)
driver.get(url)

#get html and parse with soup
doc = driver.page_source
soup = BeautifulSoup(doc, 'html.parser')
items = soup.findAll("div", class_="lister-item mode-detail")

ACTOR_ID = []
RANKING = []
NAME = []
ABOUT = []

actor_id = 0
ranking = 1
for item in items:
    name = item.find("h3").find("a")

    actor_url = name["href"]
    full_actor_url = f"https://www.imdb.com{actor_url}"
    driver.get(full_actor_url)
    doc = driver.page_source
    soup = BeautifulSoup(doc, 'html.parser')
    
    actor_about_url = soup.find("a", class_="ipc-overflowText-overlay bio-overflowtext-overlay")["href"]
    full_actor_about_url = f"https://www.imdb.com{actor_about_url}"
    driver.get(full_actor_about_url)
    doc = driver.page_source
    soup = BeautifulSoup(doc, 'html.parser')
    
    bio_paragraphs = soup.find('div', {'data-testid': 'sub-section-mini_bio'}).find("div", class_="ipc-html-content-inner-div")
    #replace a tags in bio_paragraphs with their text content
    for a_tag in bio_paragraphs.find_all('a'):
        a_tag.replace_with(a_tag.get_text())
    #replaces the double <br><br> with a normal linebreak
    for br in bio_paragraphs.find_all('br'):
        next_sibling = br.find_next_sibling()
        if next_sibling and next_sibling.name == 'br':
            br.extract()
            next_sibling.replace_with('\n')
    #concatenates all paragraphs into one text 
    text = bio_paragraphs.get_text(separator='')
    #switches to \r\n since it did not print correctly in my terminal
    text_cr = text.replace('\n', '\r\n')
    
    ABOUT.append(text_cr)
    NAME.append(name.find(string=True).replace("\n", ""))
    RANKING.append(ranking)
    ACTOR_ID.append(actor_id)
    actor_id += 1
    ranking += 1

actor_df = pd.DataFrame()
actor_df["id"] = ACTOR_ID
actor_df["ranking"] = RANKING
actor_df["name"] = NAME
actor_df["about"] = ABOUT

### Inserting the actor dataframe into the corresponding table

In [ ]:
actor_df[['id', 'ranking', 'name', 'about']].to_sql('actors', connection, index=False, if_exists='replace')

### Scraping the movies data of each actor

Seleniums webdriver module is utilized to get the source of the IMDB page, while
BeautifulSoup is used to parse the html and find all actor list items

I iterate over the list of items. In each iteration the following steps are performed:
1. Find the actor URL and get the actors page
2. Click the ‘see more’ button and wait for the page to fully load
3. Find all movie items and iterate over them:
    - Find the movies name, rating and year
    - Find the movie URL and get the movies page
    - Find the genres of the movie
    - Append actorId, movieId, name, rating, year and genres to movie_entry list
    - Increment movieId
4. Increment actorId



In [6]:
actorId = 0
movieId = 0

In [ ]:
from selenium.webdriver.chrome.options import Options
chrome_options = Options()
chrome_options.add_argument('--lang=en')

url = "https://www.imdb.com/list/ls053501318/"
driver = webdriver.Chrome(options=chrome_options)
driver.get(url)

#get html and parse with soup
doc = driver.page_source
soup = BeautifulSoup(doc, 'html.parser')
items = soup.findAll("div", class_="lister-item mode-detail")

MOVIES = []

for item in items:
    name = item.find("h3").find("a")
    actor_url = name["href"]
    full_actor_url = f"https://www.imdb.com{actor_url}"

    #get html
    driver.get(full_actor_url)

    #click the "see more" button, if present
    try:
        see_more_button = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, '//*[@id="accordion-item-actor-previous-projects"]/div/div/span/button/span'))
        )
        driver.execute_script("arguments[0].click();", see_more_button)
        time.sleep(2)
    except Exception as e:
        print(f"Error clicking 'See more': {e}")

    #parse with soup
    doc = driver.page_source
    soup = BeautifulSoup(doc, 'html.parser')
    movies = soup.findAll("li", class_="ipc-metadata-list-summary-item ipc-metadata-list-summary-item--click sc-9814c2de-3 dfZQqg")
    
    for movie in movies:
        name = movie.find("a", class_="ipc-metadata-list-summary-item__t").find(string=True).replace("\n", "")
        rating_element = movie.find("span", class_="ipc-rating-star")
        rating = rating_element.find(string=True) if rating_element else None
        year_element = movie.find("span", class_="ipc-metadata-list-summary-item__li")
        year = year_element.find(string=True) if year_element else None
        
        movie_url = movie.find("a", class_="ipc-metadata-list-summary-item__t")["href"]
        full_movie_url = f"https://www.imdb.com{movie_url}"
        driver.get(full_movie_url)
        doc = driver.page_source
        soup = BeautifulSoup(doc, 'html.parser')
        genre_items_element = soup.find("div", class_="ipc-chip-list__scroller")
        genre_items = genre_items_element.findAll("span", class_="ipc-chip__text") if genre_items_element else None
        genres = []
        if genre_items != None:  
            for genre_item in genre_items:
                genres.append(genre_item.find(string=True))
        
        movie_entry = (actorId, movieId, name, rating, year, genres)
        print(movie_entry)
        MOVIES.append(movie_entry)
        movieId += 1
    actorId += 1

### Creating and populating movies, genres and actors_play_movies dataframes

In [12]:
movies_columns = ['id', 'name', 'rating', 'year']
movies_df = pd.DataFrame([(movie[1], movie[2], movie[3], movie[4]) for movie in MOVIES], columns=movies_columns)

genres_list = [(movie[1], genre) for movie in MOVIES for genre in movie[5]]
genres_columns = ['movieId', 'genre']
genres_df = pd.DataFrame(genres_list, columns=genres_columns)

actors_play_movies_columns = ['actorId', 'movieId']
actors_play_movies_df = pd.DataFrame([(movie[0], movie[1]) for movie in MOVIES], columns=actors_play_movies_columns)

### Inserting movies, genres and actors_play_movies dataframes into corresponding tables

In [ ]:
movies_df.to_sql('movies', connection, index=False, if_exists='append')
genres_df.to_sql('genres', connection, index=False, if_exists='append')
actors_play_movies_df.to_sql('actors_play_movies', connection, index=False, if_exists='append')

### Scraping awards data of each actor

Seleniums webdriver module is utilized to get the source of the IMDB page, while
BeautifulSoup is used to parse the html and find all actor list items

I iterate over the list of items. In each iteration the following steps are performed:
1. Find the actor URL and get the actors page
2. Find the awards URL and get the awards page
3. Click all ‘see more’ buttons and wait for the page to fully load
4. Find all award items and iterate over them:
    - Find the awards name, year, win/nomination and category
    - Get the movieId of award
    - Append actorId, movideId, awardId, name, year, win/nomination and category to award_entry list
    - Increment awardId
5. Increment actorId

In [86]:
actorId2 = 0
awardId = 0

In [ ]:
from selenium.webdriver.chrome.options import Options
chrome_options = Options()
chrome_options.add_argument('--lang=en')

url = "https://www.imdb.com/list/ls053501318/"
driver = webdriver.Chrome(options=chrome_options)
driver.get(url)

#get html and parse with soup
doc = driver.page_source
soup = BeautifulSoup(doc, 'html.parser')
items = soup.findAll("div", class_="lister-item mode-detail")

AWARDS = []

for item in items:
    name = item.find("h3").find("a")
    actor_url = name["href"]
    full_actor_url = f"https://www.imdb.com{actor_url}"

    #get html and parse with soup
    driver.get(full_actor_url)
    doc = driver.page_source
    soup = BeautifulSoup(doc, 'html.parser')

    awards_item = soup.find('div', {'data-testid': 'awards'})
    awards_url = awards_item.find("a", class_="ipc-metadata-list-item__icon-link")["href"]
    full_awards_url = f"https://www.imdb.com{awards_url}"
    driver.get(full_awards_url)

    #locate see more buttons after waiting for them to load
    time.sleep(2)
    xpath_expression = '//div[contains(@class, "sc-2e6342b6-0") and contains(@class, "bclijB") and contains(@class, "pagination-container")]'
    see_more_buttons = driver.find_elements(By.XPATH, xpath_expression)
    number_of_buttons = len(see_more_buttons)
    print(number_of_buttons)

    #iterate over "see more" buttons
    for button in see_more_buttons:
        try:
            #find the first element with class "ipc-see-more__text" within the current button
            see_more_text_element = button.find_element(By.CLASS_NAME, "ipc-see-more__text")

            #click the "see more" text element
            driver.execute_script("arguments[0].click();", see_more_text_element)

            #wait for the page to load after clicking
            time.sleep(1)

        except Exception as e:
            print(f"Error: {e}")
         
    doc = driver.page_source
    soup = BeautifulSoup(doc, 'html.parser')
    awards = soup.findAll("li", class_="ipc-metadata-list-summary-item sc-15fc9ae6-1 kZSOHj")

    for award in awards:
        name = award.find("span", class_="ipc-metadata-list-summary-item__tst").find(string=True)
        year_and_win = award.find("a", class_="ipc-metadata-list-summary-item__t").find(string=True).split()
        year = year_and_win[0]
        win = year_and_win[1]
        category_element = award.find("span", class_="ipc-metadata-list-summary-item__li awardCategoryName")
        category = category_element.find(string=True) if category_element else None
        movie_element = award.find("a", class_="ipc-metadata-list-summary-item__li ipc-metadata-list-summary-item__li--link")
        movie = movie_element.find(string=True) if movie_element else None

        movieId = None
        if movie!= None:
            query = "SELECT id FROM movies WHERE name=?"
            movieIds = execute_read_query_parameterized(connection, query, movie)
            
            if movieIds != []:
                movieId = movieIds[0][0]
                movieId = int(movieId)

        award_entry = (actorId2, movieId, awardId, name, year, win, category)
        AWARDS.append(award_entry)
        awardId+=1
    print("actor: " + str(actorId2))
    actorId2+=1
AWARDS

### Creating and populating awards dataframe

In [ ]:
awards_columns = ['actorId', 'movieId', 'id', 'name', 'year', 'win', 'category']
awards_df = pd.DataFrame([(award[0], award[1], award[2], award[3], award[4], award[5], award[6]) for award in AWARDS], columns=awards_columns)

### Inserting awards dataframe into corresponding table

In [ ]:
awards_df.to_sql('awards', connection, index=False, if_exists='replace')

### Removing the leading whitespace from the names

In [ ]:
execute_query(connection, "UPDATE awards SET name = LTRIM(name);")

In [ ]:
execute_query(connection, "UPDATE actors SET name = LTRIM(name);")

### Closing the database connection

In [ ]:
connection.close()